# Unidad 9 — Bases de Datos con SQLite

**Programación Orientada a Objetos — ISTEA**

SQLite es una base de datos relacional que no requiere servidor — ideal para aprender SQL
y para aplicaciones pequeñas. Python incluye el módulo `sqlite3` en la biblioteca estándar.

**Temas que practico en esta unidad:**
- Usar SQL básico: CREATE TABLE, INSERT, SELECT, UPDATE, DELETE
- Conectarse a SQLite con sqlite3
- Usar consultas parametrizadas para evitar SQL injection
- Ejecutar múltiples inserciones con executemany
- Aplicar WHERE, ORDER BY, GROUP BY, JOIN
- Usar fetchall, fetchone, rowcount

*Consignas de la cátedra. Resoluciones desarrolladas durante la cursada.*

## Ejercicios

### 🟢 Ejercicio 1 — Crear tabla e insertar con executemany
Crear tabla "productos" e insertar 5 registros.


In [1]:
# 🎯 EJERCICIO 1: Tabla productos con executemany
import sqlite3

conn_ej = sqlite3.connect(":memory:")  # base en memoria, descartable
cursor_ej = conn_ej.cursor()

# 1. CREATE TABLE productos (id, nombre, precio, categoria, stock)
cursor_ej.execute("""
    CREATE TABLE productos (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT NOT NULL,
        precio REAL NOT NULL,
        categoria TEXT NOT NULL,
        stock INTEGER NOT NULL
    )
""")

print("Tabla 'productos' creada correctamente")

# Datos a insertar
productos_data = [
    ("Laptop", 350000, "Electrónica", 10),
    ("Mouse", 8500, "Electrónica", 50),
    ("Silla ergonómica", 45000, "Muebles", 15),
    ("Escritorio", 85000, "Muebles", 8),
    ("Auriculares", 22000, "Electrónica", 30),
]

# 2. Insertar 5 productos con executemany
cursor_ej.executemany("""
    INSERT INTO productos (nombre, precio, categoria, stock)
    VALUES (?, ?, ?, ?)
""", productos_data)

conn_ej.commit()

# Verificar inserción
cursor_ej.execute("SELECT * FROM productos")
for fila in cursor_ej.fetchall():
    print(fila)

Tabla 'productos' creada correctamente
(1, 'Laptop', 350000.0, 'Electrónica', 10)
(2, 'Mouse', 8500.0, 'Electrónica', 50)
(3, 'Silla ergonómica', 45000.0, 'Muebles', 15)
(4, 'Escritorio', 85000.0, 'Muebles', 8)
(5, 'Auriculares', 22000.0, 'Electrónica', 30)


### 🟡 Ejercicio 2 — SELECT con WHERE + ORDER BY + LIMIT
Consultar productos con precio > 20000, ordenados por precio descendente, primeros 3.


In [2]:
# 🎯 EJERCICIO 2: SELECT con filtros
# Usando la conexión del ejercicio anterior (conn_ej)

cursor_ej.execute("""
    SELECT nombre, precio, categoria
    FROM productos
    WHERE precio > 20000
    ORDER BY precio DESC
    LIMIT 3
""")

print("--- Los 3 productos más caros con precio > 20000 ---")
for producto in cursor_ej.fetchall():
    print(producto)

--- Los 3 productos más caros con precio > 20000 ---
('Laptop', 350000.0, 'Electrónica')
('Escritorio', 85000.0, 'Muebles')
('Silla ergonómica', 45000.0, 'Muebles')


### 🟡 Ejercicio 3 — UPDATE y verificación
Actualizar el precio de "Mouse" a 9500 y verificar el cambio.


In [3]:
# 🎯 EJERCICIO 3: UPDATE y verificación

# Actualizar el precio de "Mouse" a 9500
cursor_ej.execute("""
    UPDATE productos
    SET precio = 9500
    WHERE nombre = 'Mouse'
""")
conn_ej.commit()

# SELECT para verificar el cambio
cursor_ej.execute("SELECT * FROM productos WHERE nombre = 'Mouse'")
print("--- Verificación del UPDATE (Mouse) ---")
print(cursor_ej.fetchone())

--- Verificación del UPDATE (Mouse) ---
(2, 'Mouse', 9500.0, 'Electrónica', 50)


### 🟡 Ejercicio 4 — DELETE con condición
Eliminar todos los productos de categoría "Muebles".


In [4]:
# 🎯 EJERCICIO 4: DELETE con condición

# Eliminar todos los productos de categoría "Muebles"
cursor_ej.execute("""
    DELETE FROM productos
    WHERE categoria = 'Muebles'
""")
conn_ej.commit()

# SELECT para verificar la eliminación de "Muebles" y ver los remanentes
cursor_ej.execute("SELECT * FROM productos")
print("--- Productos remanentes tras DELETE de Muebles ---")
for producto in cursor_ej.fetchall():
    print(producto)

conn_ej.close()

--- Productos remanentes tras DELETE de Muebles ---
(1, 'Laptop', 350000.0, 'Electrónica', 10)
(2, 'Mouse', 9500.0, 'Electrónica', 50)
(5, 'Auriculares', 22000.0, 'Electrónica', 30)


### 🔴 Ejercicio 5 — JOIN: películas con rating promedio
JOIN entre peliculas y ratings para mostrar cada película con su promedio y cantidad de votos.


In [5]:
# 🎯 EJERCICIO 5: JOIN completo
import sqlite3

# Usamos :memory: para que sea autoejecutable, simulando DB_PATH
conn2 = sqlite3.connect(":memory:")
conn2.row_factory = sqlite3.Row
cursor2 = conn2.cursor()

# --- Configuración del entorno de prueba (Tablas ficticias) ---
cursor2.execute("""
    CREATE TABLE peliculas (
        id INTEGER PRIMARY KEY,
        titulo TEXT NOT NULL
    )
""")
cursor2.execute("""
    CREATE TABLE ratings (
        id INTEGER PRIMARY KEY,
        pelicula_id INTEGER,
        rating REAL,
        FOREIGN KEY (pelicula_id) REFERENCES peliculas(id)
    )
""")

# Inserción de datos mínimos para el test
cursor2.execute("INSERT INTO peliculas (titulo) VALUES ('Inception'), ('Interstellar'), ('The Dark Knight')")
cursor2.executemany("INSERT INTO ratings (pelicula_id, rating) VALUES (?, ?)", [
    (1, 5.0), (1, 4.5), (1, 4.0),
    (2, 5.0), (2, 5.0),
    (3, 4.0)
])
conn2.commit()
# -----------------------------------------------------------------

# CONSULTA CON JOIN, AVG Y COUNT
query = """
    SELECT p.titulo,
           AVG(r.rating) AS promedio_rating,
           COUNT(r.id) AS cantidad_votos
    FROM peliculas p
    INNER JOIN ratings r ON p.id = r.pelicula_id
    GROUP BY p.id, p.titulo
    ORDER BY promedio_rating DESC
"""

cursor2.execute(query)

print(f"{'Película':<20} | {'Promedio':<8} | {'Votos':<5}")
print("-" * 40)
for row in cursor2.fetchall():
    # Al usar row_factory = sqlite3.Row, podemos acceder por nombre de columna
    print(f"{row['titulo']:<20} | {row['promedio_rating']:<8.2f} | {row['cantidad_votos']:<5}")

conn2.close()

Película             | Promedio | Votos
----------------------------------------
Interstellar         | 5.00     | 2    
Inception            | 4.50     | 3    
The Dark Knight      | 4.00     | 1    


## Desafío extra 🔴

Implementar una clase `BaseDatos` que encapsule la conexión a SQLite y exponga:
- `crear_tabla(nombre, columnas_dict)`: crea tabla dinámicamente
- `insertar(tabla, datos_dict)`: inserta un registro
- `consultar(tabla, where=None, order_by=None, limit=None)`: SELECT con filtros opcionales
- `actualizar(tabla, datos, where)`: UPDATE
- `eliminar(tabla, where)`: DELETE

Usar la clase para gestionar una tabla de "contactos".


In [6]:
# 🔴 Desafío: Clase BaseDatos
import sqlite3

class BaseDatos:
    def __init__(self, ruta=":memory:"):
        self.conn = sqlite3.connect(ruta)
        self.conn.row_factory = sqlite3.Row
        self.cursor = self.conn.cursor()

    # Nota: en SQLite los VALORES se pasan con "?" (evita SQL injection), pero los nombres
    # de tablas y columnas no se pueden parametrizar, por eso van en el texto de la consulta.

    def crear_tabla(self, nombre, columnas_dict):
        columnas = ", ".join(f"{col} {tipo}" for col, tipo in columnas_dict.items())
        self.cursor.execute(f"CREATE TABLE IF NOT EXISTS {nombre} (id INTEGER PRIMARY KEY AUTOINCREMENT, {columnas})")
        self.conn.commit()

    def insertar(self, tabla, datos_dict):
        campos = ", ".join(datos_dict)
        marcadores = ", ".join("?" for _ in datos_dict)
        self.cursor.execute(f"INSERT INTO {tabla} ({campos}) VALUES ({marcadores})", tuple(datos_dict.values()))
        self.conn.commit()
        return self.cursor.lastrowid

    def _armar_where(self, where):
        if not where:
            return "", ()
        condicion = " AND ".join(f"{col} = ?" for col in where)
        return f" WHERE {condicion}", tuple(where.values())

    def consultar(self, tabla, where=None, order_by=None, limit=None):
        clausula, parametros = self._armar_where(where)
        sql = f"SELECT * FROM {tabla}{clausula}"
        if order_by:
            sql += f" ORDER BY {order_by}"
        if limit:
            sql += f" LIMIT {int(limit)}"
        return [dict(fila) for fila in self.cursor.execute(sql, parametros).fetchall()]

    def actualizar(self, tabla, datos, where):
        cambios = ", ".join(f"{col} = ?" for col in datos)
        clausula, parametros = self._armar_where(where)
        self.cursor.execute(f"UPDATE {tabla} SET {cambios}{clausula}", tuple(datos.values()) + parametros)
        self.conn.commit()
        return self.cursor.rowcount

    def eliminar(self, tabla, where):
        clausula, parametros = self._armar_where(where)
        self.cursor.execute(f"DELETE FROM {tabla}{clausula}", parametros)
        self.conn.commit()
        return self.cursor.rowcount

    def cerrar(self):
        self.conn.close()

# ─── Uso: tabla de contactos ───
bd = BaseDatos()
bd.crear_tabla("contactos", {"nombre": "TEXT", "email": "TEXT", "telefono": "TEXT"})

for contacto in [
    {"nombre": "Ana", "email": "ana@mail.com", "telefono": "011-1234"},
    {"nombre": "Bruno", "email": "bruno@mail.com", "telefono": "0348-5678"},
    {"nombre": "Carla", "email": "carla@mail.com", "telefono": "011-9012"},
]:
    bd.insertar("contactos", contacto)

print("Todos, ordenados por nombre:")
for c in bd.consultar("contactos", order_by="nombre"):
    print(" ", c)

print("\nActualizo el teléfono de Bruno:", bd.actualizar("contactos", {"telefono": "0348-0000"}, {"nombre": "Bruno"}), "fila")
print(bd.consultar("contactos", where={"nombre": "Bruno"}))

print("\nElimino a Carla:", bd.eliminar("contactos", {"nombre": "Carla"}), "fila")
print("Quedan:", [c["nombre"] for c in bd.consultar("contactos")])
bd.cerrar()

Todos, ordenados por nombre:
  {'id': 1, 'nombre': 'Ana', 'email': 'ana@mail.com', 'telefono': '011-1234'}
  {'id': 2, 'nombre': 'Bruno', 'email': 'bruno@mail.com', 'telefono': '0348-5678'}
  {'id': 3, 'nombre': 'Carla', 'email': 'carla@mail.com', 'telefono': '011-9012'}

Actualizo el teléfono de Bruno: 1 fila
[{'id': 2, 'nombre': 'Bruno', 'email': 'bruno@mail.com', 'telefono': '0348-0000'}]

Elimino a Carla: 1 fila
Quedan: ['Ana', 'Bruno']


## Resumen de conceptos

| SQL | Descripción |
|---|---|
| `CREATE TABLE` | Crear tabla con columnas y tipos |
| `INSERT INTO ... VALUES (?)` | Insertar fila (usar ? para parámetros) |
| `executemany(sql, lista)` | Insertar múltiples filas |
| `SELECT ... FROM ... WHERE` | Consultar con filtros |
| `ORDER BY col DESC/ASC` | Ordenar resultados |
| `LIMIT n` | Limitar cantidad de resultados |
| `UPDATE ... SET ... WHERE` | Actualizar registros |
| `DELETE FROM ... WHERE` | Eliminar registros |
| `GROUP BY` | Agrupar para agregar |
| `HAVING` | Filtrar grupos |
| `INNER JOIN ... ON` | Unir tablas por clave foránea |
| `fetchall()` / `fetchone()` | Obtener resultados |
